# Step 0 — Search strategy and validation

## 1. Objective and scope

This notebook develops and validates a patent search strategy for
precision agriculture in crop production, with John Deere as the client.

The search strategy is primarily informed by Tey et al. (2024),
*Evolution of precision agricultural technologies: a patent network analysis*.

We use the agricultural and technological terms reported in Table 1
as our starting point. Our adaptations, including specific phrases,
CPC codes and any exclusion rules, are documented separately.

The scope covers technologies that support crop monitoring,
agricultural decision-making and targeted or automated crop operations.
Livestock-only applications, forestry and lawn care are outside
our chosen scope.

The unit of analysis is the DOCDB patent family.
Search results are treated as candidate families whose relevance
must be assessed through validation.

## Search routes

A family is selected if it satisfies at least one of three routes.

**Route 1 — Specific CPC/IPC**: At least one application has a selected specific CPC or IPC code.
No keyword match is required. The specific IPC list is currently empty, so this route currently
relies on the selected specific CPC codes.

**Route 2 — Specific keywords**: A title or abstract contains an explicit expression such as
"precision agriculture" or "precision farming".


**Route 3 — Keywords supported by agricultural classification**: All three conditions must be satisfied:

1. An agricultural keyword is present.
2. A technological keyword is present.
3. An agricultural CPC or IPC code is present.

Agricultural and technological keywords alone are therefore insufficient for this route.

### Overall rule

$$
\mathrm{Precision Agriculture} =
\mathrm{CPC/IPC}_{specific}
\cup
\mathrm{Keyword}_{specific}
\cup
\left(
\mathrm{Keyword}_{agri}
\cap
\mathrm{Keyword}_{tech}
\cap
\mathrm{CPC/IPC}_{agri}
\right)
$$


The unit of analysis is the DOCDB family. Evidence is combined across the family's applications of the
selected IP right type. Different applications may provide the keyword and classification matches.
A family is counted once even when it satisfies several routes.

The time window uses the minimum `earliest_filing_year` among
these applications. This is not explicitly an earliest-priority-date
calculation.


## 2. Imports and Configuration Load

We load the search terms and analysis settings from the YAML file.
The analysis covers families with an earliest filing year between
2000 and 2022, inclusive. We use PATSTAT's `earliest_filing_year`
as the time variable.

In [1]:
from pathlib import Path
import re

import pandas as pd
import yaml

from sqlalchemy import and_, or_, func, false

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS202_APPLN_TITLE,
    TLS203_APPLN_ABSTR,
    TLS209_APPLN_IPC,
    TLS224_APPLN_CPC,
)

In [2]:
# Locate the project from either its root or the notebooks folder.
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "config" / "sector.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "config" / "sector.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Configuration not found: {CONFIG_PATH}")

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

ENV = config["patstat"]["environment"]
IPR_TYPE = config["patstat"]["ipr_type"]

YEAR_MIN = int(config["analysis"]["year_min"])
YEAR_MAX = int(config["analysis"]["year_max"])
FAMILY_TIME_COLUMN = config["analysis"]["family_time_column"]

SPECIFIC_PHRASES = config["keywords"]["specific"]
AGRICULTURAL_TERMS = config["keywords"]["agricultural"]
TECHNOLOGICAL_TERMS = config["keywords"]["technological"]

SPECIFIC_CPC = config["cpc"]["specific"]
AGRICULTURAL_CPC = config["cpc"]["agricultural_context"]

SPECIFIC_IPC = config["ipc"]["specific"]
AGRICULTURAL_IPC = config["ipc"]["agricultural_context"]

SAMPLE_SIZE = int(config["validation"]["sample_size"])
RANDOM_SEED = int(config["validation"]["random_seed"])

if YEAR_MIN > YEAR_MAX:
    raise ValueError("year_min must not exceed year_max.")

if FAMILY_TIME_COLUMN != "earliest_filing_year":
    raise ValueError("This notebook uses earliest_filing_year.")

print("Configuration:", CONFIG_PATH)
print("Environment:", ENV)
print(f"Period: {YEAR_MIN}–{YEAR_MAX}")

Configuration: /home/jovyan/team-38-PA/config/sector.yaml
Environment: PROD
Period: 2000–2022


## Prepare matching conditions

Keywords are searched in lowercase titles and abstracts.

We interpret `*` as a word-ending wildcard, accept spaces or
hyphens between words, and use word boundaries to avoid matching
abbreviations inside unrelated words.

We adapt the paper's `Automat control*` to `Automat* control*`
to match expressions such as "automatic control".


In [3]:
def term_to_regex(term):
    """Convert a search term into a word-bounded regex."""
    term = term.strip().lower()

    if not term:
        raise ValueError("Search terms cannot be empty.")

    parts = re.split(r"[\s-]+", term)

    patterns = [
        re.escape(part).replace(r"\*", r"\w*")
        for part in parts
    ]

    return r"\b" + r"[\s-]+".join(patterns) + r"\b"


def keyword_condition(patterns):
    """Match any pattern in the combined title and abstract."""
    if not patterns:
        return false()

    doc_text = func.lower(
        func.concat(
            func.coalesce(TLS202_APPLN_TITLE.appln_title, ""),
            " ",
            func.coalesce(TLS203_APPLN_ABSTR.appln_abstract, ""),
        )
    )

    combined_pattern = "|".join(
        f"(?:{pattern})" for pattern in patterns.values()
    )

    return func.regexp_contains(doc_text, combined_pattern)


def symbol_prefix_condition(symbol_column, prefixes):
    """Match any classification prefix after removing spaces."""
    if not prefixes:
        return false()

    symbol = func.upper(func.replace(symbol_column, " ", ""))

    return or_(
        *(
            symbol.like(prefix.replace(" ", "").upper() + "%")
            for prefix in prefixes
        )
    )


SPECIFIC_PATTERNS = {
    term: term_to_regex(term)
    for term in SPECIFIC_PHRASES
}

AGRICULTURAL_PATTERNS = {
    term: term_to_regex(term)
    for term in AGRICULTURAL_TERMS
}

TECHNOLOGICAL_PATTERNS = {
    term: term_to_regex(term)
    for term in TECHNOLOGICAL_TERMS
}

## Connect to PATSTAT

We use SQLAlchemy to build queries and the PATSTAT client
to execute them.

In [4]:
patstat = PatstatClient(env=ENV)
db = patstat.orm()

print(f"PATSTAT client initialized: {ENV}")

PATSTAT client initialized: PROD


## Define the family population

We select positive DOCDB family identifiers and restrict the
applications to the configured IP right type.

The family's minimum earliest filing year must fall within
the configured period, including both endpoints.

In [5]:
family_year = func.min(TLS201_APPLN.earliest_filing_year)

family_scope = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),
        family_year.label("family_year"),
    )
    .filter(
        TLS201_APPLN.ipr_type == IPR_TYPE,
        TLS201_APPLN.docdb_family_id > 0,
    )
    .group_by(TLS201_APPLN.docdb_family_id)
    .having(family_year.between(YEAR_MIN, YEAR_MAX))
    .subquery()
)

## Collect keyword evidence

For each family, we check whether any eligible application
matches each keyword group.

Missing titles and abstracts are treated as empty text.
The English search terms are applied to the available text;
we do not translate non-English records.

The agricultural and technological matches may occur in
different applications within the same family.

In [6]:
keyword_flags = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),
        func.logical_or(
            keyword_condition(SPECIFIC_PATTERNS)
        ).label("specific_phrase"),
        func.logical_or(
            keyword_condition(AGRICULTURAL_PATTERNS)
        ).label("agricultural_keyword"),
        func.logical_or(
            keyword_condition(TECHNOLOGICAL_PATTERNS)
        ).label("technological_keyword"),
    )
    .select_from(TLS201_APPLN)
    .join(
        family_scope,
        TLS201_APPLN.docdb_family_id == family_scope.c.fam_id,
    )
    .outerjoin(
        TLS202_APPLN_TITLE,
        TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
    )
    .outerjoin(
        TLS203_APPLN_ABSTR,
        TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
    )
    .filter(TLS201_APPLN.ipr_type == IPR_TYPE)
    .group_by(TLS201_APPLN.docdb_family_id)
    .subquery()
)

## Collect classification evidence

CPC and IPC evidence is aggregated separately from text evidence
to avoid multiplying text rows by classification matches.

For each classification system, we record whether the family
matches a specific code or an agricultural-context code.

A family without classification evidence can still qualify
through the specific-phrase route.

In [7]:
cpc_flags = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),
        func.logical_or(
            symbol_prefix_condition(
                TLS224_APPLN_CPC.cpc_class_symbol,
                SPECIFIC_CPC,
            )
        ).label("specific_cpc"),
        func.logical_or(
            symbol_prefix_condition(
                TLS224_APPLN_CPC.cpc_class_symbol,
                AGRICULTURAL_CPC,
            )
        ).label("agricultural_cpc"),
    )
    .select_from(TLS201_APPLN)
    .join(
        family_scope,
        TLS201_APPLN.docdb_family_id == family_scope.c.fam_id,
    )
    .join(
        TLS224_APPLN_CPC,
        TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id,
    )
    .filter(TLS201_APPLN.ipr_type == IPR_TYPE)
    .group_by(TLS201_APPLN.docdb_family_id)
    .subquery()
)

In [8]:
ipc_flags = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),
        func.logical_or(
            symbol_prefix_condition(
                TLS209_APPLN_IPC.ipc_class_symbol,
                SPECIFIC_IPC,
            )
        ).label("specific_ipc"),
        func.logical_or(
            symbol_prefix_condition(
                TLS209_APPLN_IPC.ipc_class_symbol,
                AGRICULTURAL_IPC,
            )
        ).label("agricultural_ipc"),
    )
    .select_from(TLS201_APPLN)
    .join(
        family_scope,
        TLS201_APPLN.docdb_family_id == family_scope.c.fam_id,
    )
    .join(
        TLS209_APPLN_IPC,
        TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id,
    )
    .filter(TLS201_APPLN.ipr_type == IPR_TYPE)
    .group_by(TLS201_APPLN.docdb_family_id)
    .subquery()
)

## Combine evidence and apply the inclusion rule

We join the evidence into one row per family.
Missing evidence is represented by FALSE.

- Route 1 requires a specific CPC or IPC code.
- Route 2 requires a specific phrase.
- Route 3 requires agricultural keywords, technological keywords and an agricultural CPC or IPC code.

Families satisfying any route are retained.

In [9]:
flags = (
    db.query(
        family_scope.c.fam_id,
        family_scope.c.family_year,
        func.coalesce(
            keyword_flags.c.specific_phrase, False
        ).label("specific_phrase"),
        func.coalesce(
            keyword_flags.c.agricultural_keyword, False
        ).label("agricultural_keyword"),
        func.coalesce(
            keyword_flags.c.technological_keyword, False
        ).label("technological_keyword"),
        func.coalesce(
            cpc_flags.c.specific_cpc, False
        ).label("specific_cpc"),
        func.coalesce(
            cpc_flags.c.agricultural_cpc, False
        ).label("agricultural_cpc"),
        func.coalesce(
            ipc_flags.c.specific_ipc, False
        ).label("specific_ipc"),
        func.coalesce(
            ipc_flags.c.agricultural_ipc, False
        ).label("agricultural_ipc"),
    )
    .select_from(family_scope)
    .outerjoin(
        keyword_flags,
        family_scope.c.fam_id == keyword_flags.c.fam_id,
    )
    .outerjoin(
        cpc_flags,
        family_scope.c.fam_id == cpc_flags.c.fam_id,
    )
    .outerjoin(
        ipc_flags,
        family_scope.c.fam_id == ipc_flags.c.fam_id,
    )
    .subquery()
)

route_1 = or_(
    flags.c.specific_cpc,
    flags.c.specific_ipc,
)

route_2 = flags.c.specific_phrase

route_3 = and_(
    flags.c.agricultural_keyword,
    flags.c.technological_keyword,
    or_(
        flags.c.agricultural_cpc,
        flags.c.agricultural_ipc,
    ),
)

sector_query = (
    db.query(
        *flags.c,
        route_1.label("route_1"),
        route_2.label("route_2"),
        route_3.label("route_3"),
    )
    .select_from(flags)
    .filter(or_(route_1, route_2, route_3))
)

selected_families = sector_query.subquery()

print("Sector query prepared.")

Sector query prepared.


## Execute the search and summarise retrieval routes

We count families by their combination of routes.

The combinations are mutually exclusive, so their counts sum
to the total number of selected families.


In [10]:
route_counts_query = (
    db.query(
        selected_families.c.route_1,
        selected_families.c.route_2,
        selected_families.c.route_3,
        func.count().label("n_families"),
    )
    .select_from(selected_families)
    .group_by(
        selected_families.c.route_1,
        selected_families.c.route_2,
        selected_families.c.route_3,
    )
    .order_by(func.count().desc())
)

route_counts = patstat.df(route_counts_query)

if route_counts.empty:
    total_families = 0
    route_counts = pd.DataFrame(
        columns=[
            "route_1",
            "route_2",
            "route_3",
            "n_families",
            "share_pct",
        ]
    )
else:
    total_families = int(route_counts["n_families"].sum())
    route_counts["share_pct"] = (
        100 * route_counts["n_families"] / total_families
    ).round(1)

print(f"Selected families: {total_families:,}")
route_counts

Selected families: 25,793


,route_1,route_2,route_3,n_families,share_pct
0,False,False,True,20911,81.1
1,True,False,False,2965,11.5
2,True,False,True,1557,6.0
3,False,True,False,224,0.9
4,False,True,True,82,0.3
5,True,True,True,54,0.2


# Validation
**Random sampling for manual validation**

We select 100 DOCDB families through simple random sampling
without replacement from the complete retrieved dataset.

Every family has the same probability of selection.
A fixed random seed makes the sampling reproducible when the
population and its ordering remain unchanged.

Each sampled family will be manually assessed against our scope:

- 1 = relevant
- 0 = not relevant

During manual labelling, we use the International Society of Precision
Agriculture (ISPA) definition as our reference:

> “Precision Agriculture is a management strategy that gathers, processes
> and analyzes temporal, spatial and individual plant and animal data
> and combines it with other information to support management decisions
> according to estimated variability for improved resource use efficiency,
> productivity, quality, profitability and sustainability of agricultural
> production.”

Uncertain cases will be reviewed before calculating precision.
Retrieval routes will not be shown in the
labelling sheet.


In [12]:
sampling_query = (
    db.query(selected_families.c.fam_id)
    .select_from(selected_families)
    .distinct()
)

sampling_population = patstat.df(sampling_query)

if sampling_population.empty:
    raise ValueError("No families available for sampling.")

if sampling_population["fam_id"].isna().any():
    raise ValueError("Missing family IDs found.")

# Sort before sampling to ensure a stable starting order.
sampling_population["fam_id"] = (
    sampling_population["fam_id"].astype("int64")
)

sampling_population = (
    sampling_population
    .sort_values("fam_id")
    .reset_index(drop=True)
)

print(f"Families available: {len(sampling_population):,}")

Families available: 25,793


In [14]:
if not 0 < SAMPLE_SIZE <= len(sampling_population):
    raise ValueError("Invalid sample size.")

validation_sample = (
    sampling_population
    .sample(
        n=SAMPLE_SIZE,
        replace=False,
        random_state=RANDOM_SEED,
    )
    .reset_index(drop=True)
)

assert len(validation_sample) == SAMPLE_SIZE
assert validation_sample["fam_id"].is_unique

print(f"Sampled families: {len(validation_sample)}")

Sampled families: 100


In [15]:
from sqlalchemy import case

sample_ids = [
    int(fam_id)
    for fam_id in validation_sample["fam_id"]
]

title_present = (
    func.length(
        func.trim(
            func.coalesce(TLS202_APPLN_TITLE.appln_title, "")
        )
    ) > 0
)

abstract_present = (
    func.length(
        func.trim(
            func.coalesce(TLS203_APPLN_ABSTR.appln_abstract, "")
        )
    ) > 0
)

english_title = and_(
    TLS202_APPLN_TITLE.appln_title_lg == "en",
    title_present,
)

english_abstract = and_(
    TLS203_APPLN_ABSTR.appln_abstract_lg == "en",
    abstract_present,
)

# Prefer English text, then other available text.
text_priority = case(
    (and_(english_abstract, english_title), 0),
    (english_abstract, 1),
    (and_(abstract_present, english_title), 2),
    (abstract_present, 3),
    (english_title, 4),
    (title_present, 5),
    else_=6,
)

representative_rank = func.row_number().over(
    partition_by=TLS201_APPLN.docdb_family_id,
    order_by=[
        text_priority,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.appln_id,
        func.coalesce(TLS202_APPLN_TITLE.appln_title_lg, ""),
        func.coalesce(TLS203_APPLN_ABSTR.appln_abstract_lg, ""),
        func.coalesce(TLS202_APPLN_TITLE.appln_title, ""),
        func.coalesce(TLS203_APPLN_ABSTR.appln_abstract, ""),
    ],
).label("representative_rank")

ranked_applications = (
    db.query(
        TLS201_APPLN.docdb_family_id.label("fam_id"),
        TLS201_APPLN.appln_id,
        TLS202_APPLN_TITLE.appln_title.label("title"),
        TLS203_APPLN_ABSTR.appln_abstract.label("abstract"),
        representative_rank,
    )
    .select_from(TLS201_APPLN)
    .outerjoin(
        TLS202_APPLN_TITLE,
        TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
    )
    .outerjoin(
        TLS203_APPLN_ABSTR,
        TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
    )
    .filter(
        TLS201_APPLN.docdb_family_id.in_(sample_ids),
        TLS201_APPLN.ipr_type == IPR_TYPE,
    )
    .subquery()
)

representatives_query = (
    db.query(
        ranked_applications.c.fam_id,
        ranked_applications.c.appln_id,
        ranked_applications.c.title,
        ranked_applications.c.abstract,
    )
    .select_from(ranked_applications)
    .filter(ranked_applications.c.representative_rank == 1)
)

representatives = patstat.df(representatives_query)

In [16]:
if representatives.empty:
    raise ValueError("No representative applications retrieved.")

representatives["fam_id"] = (
    representatives["fam_id"].astype("int64")
)

# Preserve the random order of the sample.
validation_sheet = (
    validation_sample[["fam_id"]]
    .merge(
        representatives,
        on="fam_id",
        how="left",
        validate="one_to_one",
        sort=False,
    )
)

if validation_sheet["appln_id"].isna().any():
    raise ValueError("Some sampled families have no representative application.")

validation_sheet["appln_id"] = (
    validation_sheet["appln_id"].astype("int64")
)

validation_sheet[["title", "abstract"]] = (
    validation_sheet[["title", "abstract"]].fillna("")
)

validation_sheet["label"] = ""

assert len(validation_sheet) == SAMPLE_SIZE
assert validation_sheet["fam_id"].is_unique

missing_abstracts = (
    validation_sheet["abstract"].str.strip().eq("").sum()
)

print(f"Families ready for labelling: {len(validation_sheet)}")
print(f"Families without an abstract: {missing_abstracts}")


Families ready for labelling: 100
Families without an abstract: 0


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "step0"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sheet_path = OUTPUT_DIR / "hand_validation_TEY.csv"

try:
    # Exclusive creation: an existing file is never overwritten.
    with sheet_path.open(
        "x",
        encoding="utf-8-sig",
        newline="",
    ) as file:
        validation_sheet.to_csv(file, index=False)

    print("CSV created:", sheet_path)

except FileExistsError:
    print("CSV already exists and was NOT overwritten:")
    print(sheet_path)